In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.options.display.max_columns = None
pd.options.display.max_rows = None


### DATA DESCRIPTION

Data available at: [tennis_wta](https://github.com/JeffSackmann/tennis_wta).  
Contains match statistics through 1968 to 2024.  
One record contains information about two players: winner and loser, match score, match duration, details about each player (like height, age, nationality) and match stats available after the mqatch finisihed (like number of double faults, number of ace's etc)

In [35]:
df = pd.read_csv("/Users/martapriv/Documents/PUM/inzynierka_code/Tennis_Scores_Predictions/data/01_raw/combined_matches.csv", index_col=0)

/var/folders/hd/nmlhh_n952v82g530l_c0x_00000gr/T/ipykernel_96270/3188696333.py:1: DtypeWarning: Columns (4,9,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/martapriv/Documents/PUM/inzynierka_code/Tennis_Scores_Predictions/data/01_raw/combined_matches.csv", index_col=0)


In [36]:
df.shape

(158092, 49)

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 158092 entries, 0 to 158091
Data columns (total 49 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   tourney_id          158092 non-null  object 
 1   tourney_name        158092 non-null  object 
 2   surface             153253 non-null  object 
 3   draw_size           137664 non-null  object 
 4   tourney_level       158092 non-null  object 
 5   tourney_date        158092 non-null  int64  
 6   match_num           158092 non-null  int64  
 7   winner_id           158092 non-null  int64  
 8   winner_seed         61072 non-null   object 
 9   winner_entry        13392 non-null   object 
 10  winner_name         158092 non-null  object 
 11  winner_hand         158092 non-null  object 
 12  winner_ht           116667 non-null  float64
 13  winner_ioc          158062 non-null  object 
 14  winner_age          155420 non-null  float64
 15  loser_id            158092 non-null  in

Available data we can divide into two categories: data available before the beginning of the match and data available after match finished. Because the goal of this work is to predict the outcome of the upcoming match, I will focus on describing data available before the beginning of the match.

### DATA AVAILABLE BEFORE THE BEGINNING OF THE MATCH

In [38]:
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,1968-W-OL-MEX-01A-1968,Guadalajara Olympics Demo,Clay,13.0,O,19681014,1,200781,NaN,NaN,Edda Buding,R,NaN,FRG,31.9,200795,NaN,NaN,Patricia Montano,U,NaN,MEX,16.1,7-5 2-6 6-4,3,R16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1968-W-OL-MEX-01A-1968,Guadalajara Olympics Demo,Clay,13.0,O,19681014,2,202512,NaN,NaN,Lucia Gongora,U,NaN,MEX,21.9,200862,NaN,NaN,Valerie Ziegenfuss,R,173.0,USA,19.2,6-1 6-2,3,R16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1968-W-OL-MEX-01A-1968,Guadalajara Olympics Demo,Clay,13.0,O,19681014,3,200869,NaN,NaN,Maria Eugenia Guzman,U,NaN,ECU,23.0,200863,NaN,NaN,Zaiga Yansone,U,NaN,URS,17.7,6-2 6-2,3,R16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1968-W-OL-MEX-01A-1968,Guadalajara Olympics Demo,Clay,13.0,O,19681014,4,200274,NaN,NaN,Julie Heldman,R,170.0,USA,22.8,200834,NaN,NaN,Suzana Gesteira,U,NaN,BRA,21.0,6-1 6-2,3,R16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1968-W-OL-MEX-01A-1968,Guadalajara Olympics Demo,Clay,13.0,O,19681014,5,200791,NaN,NaN,Rosie Reyes,R,NaN,FRA,29.5,202513,NaN,NaN,Ana Maria Ycaza,U,NaN,ECU,17.4,6-1 6-1,3,R16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
columns_before_match = ['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level',
       'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry',
       'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age',
       'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand',
       'loser_ht', 'loser_ioc', 'loser_age', 'round',
       'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points']

In [40]:
df_bm = df.copy()
df_bm = df[columns_before_match]
df_bm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 158092 entries, 0 to 158091
Data columns (total 28 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   tourney_id          158092 non-null  object 
 1   tourney_name        158092 non-null  object 
 2   surface             153253 non-null  object 
 3   draw_size           137664 non-null  object 
 4   tourney_level       158092 non-null  object 
 5   tourney_date        158092 non-null  int64  
 6   match_num           158092 non-null  int64  
 7   winner_id           158092 non-null  int64  
 8   winner_seed         61072 non-null   object 
 9   winner_entry        13392 non-null   object 
 10  winner_name         158092 non-null  object 
 11  winner_hand         158092 non-null  object 
 12  winner_ht           116667 non-null  float64
 13  winner_ioc          158062 non-null  object 
 14  winner_age          155420 non-null  float64
 15  loser_id            158092 non-null  in

In [41]:
# float64 datatype
df_bm_float_columns = df_bm.select_dtypes(include='float64').columns
df_bm[df_bm[df_bm_float_columns].notna().all(axis=1)][df_bm_float_columns].head()

,winner_ht,winner_age,loser_ht,loser_age,winner_rank,winner_rank_points,loser_rank,loser_rank_points
53417,169.0,21.0,169.0,21.5,24.0,29.0,255.0,1.0
53425,170.0,24.2,168.0,15.6,25.0,29.0,49.0,16.0
53427,175.0,18.1,175.0,17.1,2.0,247.0,8.0,72.0
53445,169.0,21.0,168.0,19.7,24.0,29.0,35.0,21.0
53447,168.0,32.5,165.0,19.7,3.0,171.0,31.0,23.0


In [42]:
# int64 datatype
df_bm_int_columns = df_bm.select_dtypes(include='int64').columns
df_bm[df_bm[df_bm_int_columns].notna().all(axis=1)][df_bm_int_columns].head()

,tourney_date,match_num,winner_id,loser_id
0,19681014,1,200781,200795
1,19681014,2,202512,200862
2,19681014,3,200869,200863
3,19681014,4,200274,200834
4,19681014,5,200791,202513


In [43]:
# object datatype
df_bm_object_columns = df_bm.select_dtypes(include='object').columns
df_bm[df_bm[df_bm_object_columns].notna().all(axis=1)][df_bm_object_columns].head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,winner_seed,winner_entry,winner_name,winner_hand,winner_ioc,loser_seed,loser_entry,loser_name,loser_hand,loser_ioc,round
119303,2010-W-INT-MEX-02A-2010,Monterrey,Hard,32,I,2.0,WC,Daniela Hantuchova,R,SVK,4.0,WC,Dominika Cibulkova,R,SVK,SF
120512,2010-W-PR-USA-05A-2010,New Haven,Hard,30,P,8.0,WC,Nadia Petrova,R,RUS,2.0,WC,Samantha Stosur,R,AUS,QF
121504,2011-W-INT-AUT-02A-2011,Linz,Hard,32,I,1.0,WC,Petra Kvitova,L,CZE,3.0,WC,Jelena Jankovic,R,SRB,SF
132816,2015-W-INT-HKG-01A-2015,Hong Kong,Hard,32,I,4.0,WC,Jelena Jankovic,R,SRB,2.0,WC,Angelique Kerber,L,GER,F


### DATA AVAILABLE AFTER THE END OF THE MATCH

In [47]:
columns_after_match = [col for col in df.columns if col not in columns_before_match]

In [49]:
print(columns_after_match)

['score', 'best_of', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced']


In [ ]:
df_am = df.copy()
df_am = df_am[columns_after_match]
df_am[df_am[columns_after_match].notna().all(axis=1)][columns_after_match].head()

,score,best_of,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced
129394,6-4 6-4,3,101.0,0.0,1.0,76.0,57.0,35.0,6.0,10.0,4.0,11.0,0.0,1.0,60.0,37.0,20.0,8.0,10.0,6.0,13.0
129395,6-3 2-6 6-3,3,131.0,1.0,5.0,76.0,53.0,34.0,7.0,13.0,5.0,9.0,2.0,8.0,100.0,61.0,36.0,15.0,13.0,6.0,21.0
129396,6-3 7-6(5),3,105.0,5.0,4.0,79.0,49.0,34.0,13.0,11.0,3.0,9.0,0.0,4.0,71.0,53.0,28.0,9.0,10.0,4.0,9.0
129397,6-4 6-3,3,109.0,0.0,3.0,67.0,37.0,26.0,16.0,9.0,1.0,3.0,2.0,7.0,80.0,45.0,27.0,15.0,10.0,4.0,17.0
129398,7-6(4) 6-2,3,134.0,5.0,8.0,79.0,49.0,31.0,12.0,10.0,5.0,15.0,0.0,2.0,86.0,64.0,30.0,6.0,10.0,7.0,15.0


In [53]:
df_am.info()

<class 'pandas.core.frame.DataFrame'>
Index: 158092 entries, 0 to 158091
Data columns (total 21 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   score      158091 non-null  object 
 1   best_of    158092 non-null  int64  
 2   minutes    22127 non-null   float64
 3   w_ace      44001 non-null   float64
 4   w_df       43962 non-null   float64
 5   w_svpt     44003 non-null   float64
 6   w_1stIn    44003 non-null   float64
 7   w_1stWon   44003 non-null   float64
 8   w_2ndWon   44003 non-null   float64
 9   w_SvGms    24158 non-null   float64
 10  w_bpSaved  43998 non-null   float64
 11  w_bpFaced  43998 non-null   float64
 12  l_ace      43996 non-null   float64
 13  l_df       43962 non-null   float64
 14  l_svpt     44002 non-null   float64
 15  l_1stIn    44003 non-null   float64
 16  l_1stWon   44003 non-null   float64
 17  l_2ndWon   44003 non-null   float64
 18  l_SvGms    24158 non-null   float64
 19  l_bpSaved  44001 non-null   

In [54]:
# float64 datatype
df_am_float_columns = df_am.select_dtypes(include='float64').columns
df_am[df_am[df_am_float_columns].notna().all(axis=1)][df_am_float_columns].head()

,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced
129394,101.0,0.0,1.0,76.0,57.0,35.0,6.0,10.0,4.0,11.0,0.0,1.0,60.0,37.0,20.0,8.0,10.0,6.0,13.0
129395,131.0,1.0,5.0,76.0,53.0,34.0,7.0,13.0,5.0,9.0,2.0,8.0,100.0,61.0,36.0,15.0,13.0,6.0,21.0
129396,105.0,5.0,4.0,79.0,49.0,34.0,13.0,11.0,3.0,9.0,0.0,4.0,71.0,53.0,28.0,9.0,10.0,4.0,9.0
129397,109.0,0.0,3.0,67.0,37.0,26.0,16.0,9.0,1.0,3.0,2.0,7.0,80.0,45.0,27.0,15.0,10.0,4.0,17.0
129398,134.0,5.0,8.0,79.0,49.0,31.0,12.0,10.0,5.0,15.0,0.0,2.0,86.0,64.0,30.0,6.0,10.0,7.0,15.0


In [55]:
# int64 datatype
df_am_int_columns = df_am.select_dtypes(include='int64').columns
df_am[df_am[df_am_int_columns].notna().all(axis=1)][df_am_int_columns].head()

,best_of
0,3
1,3
2,3
3,3
4,3


In [56]:
# object datatype
df_am_object_columns = df_am.select_dtypes(include='object').columns
df_am[df_am[df_am_object_columns].notna().all(axis=1)][df_am_object_columns].head()

,score
0,7-5 2-6 6-4
1,6-1 6-2
2,6-2 6-2
3,6-1 6-2
4,6-1 6-1
